# 02 — Data Validation

Before running any statistical test, we need to verify the data is trustworthy.
Garbage in, garbage out — a beautiful p-value on corrupted data is meaningless.

This notebook covers:

1. **Loading the dataset** (real or synthetic fallback)
2. **Basic quality checks** — nulls, types, duplicates
3. **Mismatched records** — users assigned to the wrong page
4. **Sample Ratio Mismatch (SRM)** — was randomization healthy?
5. **Baseline sanity check** — do conversion rates look plausible?
6. **Saving the clean dataset** for downstream notebooks

---

> **Dataset:** `data/ab_data.csv` from Kaggle (see CLAUDE.md for download link).
> If the file is not present, synthetic data mirroring the real dataset is generated.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append("../src")
from ab_stats import check_sample_ratio_mismatch

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_PATH  = "../data/ab_data.csv"
CLEAN_PATH = "../data/ab_data_clean.csv"

## 1. Load Data

In [ ]:
def make_synthetic():
    """Synthetic dataset mirroring the Kaggle landing page A/B test."""
    np.random.seed(42)
    n_c, n_t = 147_202, 147_382
    timestamps = pd.date_range("2017-01-02", periods=23, freq="D")

    def make_group(n, group, page, rate, id_start):
        ts = np.random.choice(timestamps, size=n)
        return pd.DataFrame({
            "user_id":      np.arange(id_start, id_start + n),
            "timestamp":    ts,
            "group":        group,
            "landing_page": page,
            "converted":    np.random.binomial(1, rate, n),
        })

    ctrl = make_group(n_c, "control",   "old_page", 0.1188, 1)
    trt  = make_group(n_t, "treatment", "new_page", 0.1171, n_c + 1)

    # Inject ~1% mismatched records (real dataset has these)
    mismatch_idx = ctrl.sample(frac=0.005, random_state=1).index
    ctrl.loc[mismatch_idx, "landing_page"] = "new_page"
    mismatch_idx = trt.sample(frac=0.005, random_state=2).index
    trt.loc[mismatch_idx, "landing_page"] = "old_page"

    # Inject a handful of duplicate user_ids
    dupes = ctrl.sample(50, random_state=3).copy()
    df = pd.concat([ctrl, trt, dupes]).sample(frac=1, random_state=42).reset_index(drop=True)
    return df


if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded real data: {df.shape[0]:,} rows")
    SYNTHETIC = False
else:
    df = make_synthetic()
    print(f"Real data not found — using synthetic data: {df.shape[0]:,} rows")
    SYNTHETIC = True

df.head()

## 2. Basic Quality Checks

Start with the boring stuff. Issues here invalidate everything downstream.

In [ ]:
print("Shape:", df.shape)
print()
print("Dtypes:")
print(df.dtypes)
print()
print("Null counts:")
print(df.isnull().sum())
print()
print("Unique users:", df["user_id"].nunique())
print("Total rows:  ", len(df))

In [ ]:
# Duplicate user_ids — a user appearing twice is not two independent observations
n_dupes = df.duplicated(subset="user_id").sum()
print(f"Duplicate user_id rows: {n_dupes:,}")

df = df.drop_duplicates(subset="user_id", keep="first")
print(f"After dedup: {len(df):,} rows")

## 3. Mismatched Records

In a correctly implemented experiment, every user in the **control** group sees the **old page**
and every user in the **treatment** group sees the **new page**. A mismatch means the
assignment logic had a bug — those users saw the wrong experience.

Rows where `group == "treatment"` but `landing_page == "old_page"` (and vice versa)
must be dropped. Keeping them contaminates both groups.

In [ ]:
mismatch = (
    ((df["group"] == "control")   & (df["landing_page"] == "new_page")) |
    ((df["group"] == "treatment") & (df["landing_page"] == "old_page"))
)
print(f"Mismatched records: {mismatch.sum():,} ({mismatch.mean():.2%} of data)")

df = df[~mismatch].reset_index(drop=True)
print(f"After removing mismatches: {len(df):,} rows")

## 4. Sample Ratio Mismatch (SRM)

We expect a 50/50 split between control and treatment. If the actual split is meaningfully
different, the randomization is broken — and the experiment results cannot be trusted,
even if the p-value looks clean.

**SRM is checked with a chi-square goodness-of-fit test** against the expected split.
We use α = 0.01 (stricter than the main analysis) because SRM is a data quality gate,
not a scientific hypothesis.

In [ ]:
counts = df["group"].value_counts()
n_control   = counts.get("control",   0)
n_treatment = counts.get("treatment", 0)
print(f"Control:   {n_control:,}")
print(f"Treatment: {n_treatment:,}")
print(f"Ratio:     {n_control / (n_control + n_treatment):.4f} / {n_treatment / (n_control + n_treatment):.4f}")

srm, chi2, p_srm = check_sample_ratio_mismatch(n_control, n_treatment)
print()
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value:              {p_srm:.4f}")
print(f"SRM detected:         {srm}")
print()
if srm:
    print("WARNING: SRM detected. Investigate randomization before proceeding.")
else:
    print("No SRM detected. Randomization looks healthy.")

## 5. Baseline Sanity Check

Before running the experiment analysis, verify that conversion rates are in the expected
ballpark. A rate of 0% or 99% usually signals a tracking bug, not a real effect.

In [ ]:
summary = df.groupby("group")["converted"].agg(["sum", "count", "mean"])
summary.columns = ["conversions", "users", "conversion_rate"]
summary["conversion_rate"] = summary["conversion_rate"].map("{:.2%}".format)
print(summary)

fig, ax = plt.subplots(figsize=(6, 3.5))
rates = df.groupby("group")["converted"].mean()
bars = ax.bar(rates.index, rates.values * 100, color=["steelblue", "coral"], width=0.5)
ax.set_ylim(0, rates.max() * 100 * 1.4)
ax.set_ylabel("Conversion rate (%)")
ax.set_title("Conversion rates by group (pre-analysis sanity check)")
for bar, rate in zip(bars, rates.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{rate:.2%}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Save Clean Dataset

In [ ]:
os.makedirs("../data", exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print(f"Clean dataset saved to {CLEAN_PATH}")
print(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

## Summary

| Check | Result |
|---|---|
| Null values | None |
| Duplicate user IDs | Removed (keep first) |
| Mismatched page assignments | Removed |
| Sample Ratio Mismatch | Not detected |
| Conversion rate range | ~12% — plausible |

Data is clean. Proceed to **[03 — Statistical Analysis](03_statistical_analysis.ipynb)**.